# Build h3_density
Runs `build_h3_density()` standalone and outputs `h3_density.csv`.
Use this to inspect/test the aggregation without running the full ETL.

Run from the `server/` directory kernel (same venv as ETL).

In [1]:
import sys
from pathlib import Path

# Ensure server/ is on sys.path so db.etl.* imports resolve
SERVER_ROOT = Path.cwd()
while SERVER_ROOT.name != 'server' and SERVER_ROOT != SERVER_ROOT.parent:
    SERVER_ROOT = SERVER_ROOT.parent
if str(SERVER_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVER_ROOT))

print('SERVER_ROOT:', SERVER_ROOT)

SERVER_ROOT: c:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\server


In [2]:
from db.etl.load_places import load_places
from db.etl.build_h3_density import build_h3_density

df = load_places()

  13,092 rows loaded (12,320 open, 772 temporarily closed)


In [3]:
density_df = build_h3_density(
    df[[
        'h3_res10', 'cuisineType', 'cost', 'venueType',
        'bnormal_0', 'bnormal_1', 'bnormal_2',
        'normal_0',  'normal_1',  'normal_2',
    ]].rename(columns={
        'h3_res10':    'h3_r10',
        'cuisineType': 'cuisine_type',
        'venueType':   'venue_type',
    }).copy()
)

print(f'Total rows: {len(density_df):,}')
density_df.head()

  94,464 / 94,464 combos processed … done
Total rows: 2,123,754


,tile,resolution,cuisine_type,cost,venue_type,score_basis,confidence,score_tier,count
0,87194ad01ffffff,7,African,10+,Dine-In,0,0,0,4
1,87194ad03ffffff,7,African,10+,Dine-In,0,0,0,1
2,87194ad04ffffff,7,African,10+,Dine-In,0,0,0,2
3,87194ad05ffffff,7,African,10+,Dine-In,0,0,0,1
4,87194ad06ffffff,7,African,10+,Dine-In,0,0,0,8


In [4]:
# Quick sanity checks
print('score_basis values:', sorted(density_df['score_basis'].unique()))
print('confidence values: ', sorted(density_df['confidence'].unique()))
print('score_tier values: ', sorted(density_df['score_tier'].unique()))
print('resolution values: ', sorted(density_df['resolution'].unique()))
print('cuisine_type sample:', sorted(density_df['cuisine_type'].unique())[:6])
print('Duplicate PK rows:  ', density_df.duplicated(
    subset=['tile','resolution','cuisine_type','cost','venue_type','score_basis','confidence','score_tier']
).sum())

score_basis values: [np.int64(0), np.int64(1)]
confidence values:  [np.int64(0), np.int64(1), np.int64(2)]
score_tier values:  [np.int64(0), np.int64(2), np.int64(3), np.int64(4)]
resolution values:  [np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
cuisine_type sample: ['', 'African', 'American', 'Asian', 'Australian', 'Bakery & Pastry']
Duplicate PK rows:   0


In [5]:
OUT_PATH = SERVER_ROOT / 'out' / 'h3_density.csv'
density_df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')

Saved to c:\Users\tichen\OneDrive - Foster + Partners\Documents\Project\london-explorer\server\out\h3_density.csv
